# Queue Statistics – Summary Tables

In [1]:
import stata_setup
stata_setup.config('/Applications/StataNow', 'mp')

from environ.settings import PROJECT_ROOT
from pystata import stata
stata.run(f'cd "{PROJECT_ROOT}"')


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      StataNow 19.5
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2025 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 2-core network, expiring 31 Oct 2026
Serial number: 501909305069
  Licensed to: Yichen Luo
               UCL

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


In [ ]:
%%stata

clear all
set more off
set varabbrev off

global PROCESSED_DATA "processed_data"
global TABLES         "tables"

UsageError: Cell magic `%%stata` not found.


In [ ]:
%%stata
****************************
* Table 1: Queue Daily Summary Statistics
* Rows: queue_length, queue_steth, n_submitted, n_finalized
* Cols: N, Mean, Median, Std Dev, Min, Max
****************************

import delimited using "$PROCESSED_DATA/queue_daily.csv", varnames(1) clear

tempname fh
file open `fh' using "$TABLES/sum_queue_daily.tex", write replace

file write `fh' "{" _n
file write `fh' "\def\sym#1{\ifmmode^{#1}\else\(^{#1}\)\fi}" _n
file write `fh' "\begin{tabular}{l*{1}{cccccc}}" _n
file write `fh' "\toprule" _n
file write `fh' "Variable & N & Mean & Median & Std Dev & Min & Max \\" _n
file write `fh' "\midrule" _n

local vars queue_length queue_steth n_submitted n_finalized
local labels `" "Queue Length (requests)" "Queue stETH" "Daily Submissions" "Daily Finalizations" "'

local i = 1
foreach v of local vars {
    local lbl : word `i' of `labels'

    quietly summarize `v', detail
    local N      = r(N)
    local mean   : display %9.2f r(mean)
    local median : display %9.2f r(p50)
    local sd     : display %9.2f r(sd)
    local min    : display %9.2f r(min)
    local max    : display %9.2f r(max)

    file write `fh' "`lbl' & `N' & `mean' & `median' & `sd' & `min' & `max' \\" _n
    local ++i
}

file write `fh' "\bottomrule" _n
file write `fh' "\end{tabular}" _n
file write `fh' "}" _n
file close `fh'

In [ ]:
%%stata
****************************
* Table 2: Wait Time Summary Statistics (overall and by year)
****************************

import delimited using "$PROCESSED_DATA/queue_requests.csv", varnames(1) clear
keep if is_finalized == 1

tempname fh
file open `fh' using "$TABLES/sum_wait_time.tex", write replace

file write `fh' "{" _n
file write `fh' "\def\sym#1{\ifmmode^{#1}\else\(^{#1}\)\fi}" _n
file write `fh' "\begin{tabular}{l*{1}{cccccc}}" _n
file write `fh' "\toprule" _n
file write `fh' "Period & N & Mean (days) & Median (days) & Std Dev & P25 & P75 \\" _n
file write `fh' "\midrule" _n

* Overall row
quietly summarize wait_days, detail
local N      = r(N)
local mean   : display %6.2f r(mean)
local median : display %6.2f r(p50)
local sd     : display %6.2f r(sd)
local p25    : display %6.2f r(p25)
local p75    : display %6.2f r(p75)
file write `fh' "All & `N' & `mean' & `median' & `sd' & `p25' & `p75' \\" _n
file write `fh' "\midrule" _n

* By year
levelsof submit_year, local(years)
foreach yr of local years {
    quietly summarize wait_days if submit_year == `yr', detail
    local N      = r(N)
    local mean   : display %6.2f r(mean)
    local median : display %6.2f r(p50)
    local sd     : display %6.2f r(sd)
    local p25    : display %6.2f r(p25)
    local p75    : display %6.2f r(p75)
    file write `fh' "`yr' & `N' & `mean' & `median' & `sd' & `p25' & `p75' \\" _n
}

file write `fh' "\bottomrule" _n
file write `fh' "\end{tabular}" _n
file write `fh' "}" _n
file close `fh'